### Initialization

- Connect to `Google Drive`

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


- Create `config.yaml` and `params.yaml` files into configration directory.

In [ ]:
import os

drive_path = "/content/drive/MyDrive/Text-summerizer"

os.makedirs(f"{drive_path}/config", exist_ok=True)

config_content = f"""
artifacts_root: {drive_path}/artifacts

data_ingestion:
  root_dir: {drive_path}/artifacts/data_ingestion
  source_URL: https://github.com/Abdullah963039/datasets-reference/raw/refs/heads/master/text_summerizer.zip
  local_data_file: {drive_path}/artifacts/data_ingestion/data.zip
  unzip_dir: {drive_path}/artifacts/data_ingestion

data_preparation:
  root_dir: {drive_path}/data_preparation
  data_path: {drive_path}/data_ingestion/custom_dataset
  train_filename: MTS-Dialog-TrainingSet.csv
  validation_filename: MTS-Dialog-ValidationSet.csv
  test_filenames: ["MTS-Dialog-TestSet-1-MEDIQA-Chat-2023.csv", "MTS-Dialog-TestSet-2-MEDIQA-Sum-2023.csv"]

data_validation:
  root_dir: {drive_path}/artifacts/data_validation
  STATUS_FILE: {drive_path}/artifacts/data_validation/status.txt
  ALL_REQUIRED_FILES: ["train", "test", "validation"]

data_transformation:
  root_dir: {drive_path}/artifacts/data_transformation
  data_path: {drive_path}/artifacts/data_ingestion/samsum_dataset
  tokenizer_name: google/pegasus-cnn_dailymail

model_trainer:
  root_dir: {drive_path}/artifacts/model_trainer
  data_path: {drive_path}/artifacts/data_transformation/samsum_dataset
  model_ckpt: google/pegasus-cnn_dailymail

model_evaluation:
  root_dir: {drive_path}/artifacts/model_evaluation
  data_path: {drive_path}/artifacts/data_transformation/samsum_dataset
  model_path: {drive_path}/artifacts/model_trainer/pegasus-samsum-model
  tokenizer_path: {drive_path}/artifacts/model_trainer/tokenizer
  metric_file_name: {drive_path}/artifacts/model_evaluation/metrics.csv
"""

with open(f"{drive_path}/config/config.yaml", "w") as f:
    f.write(config_content.strip())

params_content = """
training_arguments:
  num_train_epochs: 1
  warmup_steps: 500
  per_device_train_batch_size: 1
  weight_decay: 0.01
  logging_steps: 10
  eval_strategy: steps
  eval_steps: 500
  save_steps: 1e6
  gradient_accumulation_steps: 16
"""
with open(f"{drive_path}/config/params.yaml", "w") as f:
    f.write(params_content.strip())

- Verify GPU Availability

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available.")

GPU is available!
GPU Name: Tesla T4


### Setup Process:
- Installing libraries
- Check for requirements

In [ ]:
# TODO: Install requirements

!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr ensure evaluate -q

### Utility Functions

In [ ]:
import os
from box.exceptions import BoxValueError
import yaml
from ensure import ensure_annotations
from box import ConfigBox
from pathlib import Path


@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    try:
        with open(path_to_yaml) as yaml_file:
            content = yaml.safe_load(yaml_file)
            return ConfigBox(content)
    except BoxValueError:
        raise ValueError("yaml file is empty")
    except Exception as e:
        raise e


@ensure_annotations
def create_directories(path_to_directories: list):
    for path in path_to_directories:
        os.makedirs(path, exist_ok=True)

@ensure_annotations
def get_size(path: Path) -> str:
    size_in_kb = round(os.path.getsize(path) / 1024)
    return f"~ {size_in_kb} KB"


### Constants:

In [ ]:
from pathlib import Path

CONFIG_FILE_PATH = Path(f"{drive_path}/config/config.yaml")
PARAMS_FILE_PATH = Path(f"{drive_path}/config/params.yaml")

### Project Entities:

Defines each process of project what its requirements.
```
1. Data Ingestion Entity
2. Data Preparation Entity
3. Data Validation Entity
4. Data Transformation Entity
5. Model Trainer Entity
6. Model Evaluation Entity
```

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
  root_dir: Path
  source_URL: str
  local_data_file: Path
  unzip_dir: Path

@dataclass(frozen=True)
class DataPreparationConfig:
    root_dir: Path
    data_path: Path
    train_filename: str
    validation_filename: str
    test_filenames: list[str]

@dataclass(frozen=True)
class DataValidationConfig:
  root_dir: Path
  STATUS_FILE: str
  ALL_REQUIRED_FILES: list


@dataclass(frozen=True)
class DataTransformationConfig:
  root_dir: Path
  data_path: Path
  tokenizer_name: list

@dataclass(frozen=True)
class ModelTrainerConfig:
  root_dir: Path
  data_path: Path
  model_ckpt: Path
  num_train_epochs: int
  warmup_steps: int
  per_device_train_batch_size: int
  weight_decay: float
  logging_steps: int
  eval_strategy: str
  eval_steps: int
  save_steps: float
  gradient_accumulation_steps: int

@dataclass(frozen=True)
class ModelEvaluationConfig:
  root_dir: Path
  data_path: Path
  model_path: Path
  tokenizer_path: Path
  metric_file_name: Path

### Configration Manager

Responsable for reading configrations from `/config/config.yaml` and assign each process its configration.

In [ ]:
class ConfigrationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir,
        )

        return data_ingestion_config

    def get_data_preparation_config(self) -> DataPreparationConfig:
        config = self.config.data_preparation

        create_directories([config.root_dir])

        data_preparation_config = DataPreparationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            train_filename=config.train_filename,
            validation_filename=config.validation_filename,
            test_filenames=config.test_filenames,
        )

        return data_preparation_config

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES,
        )

        return data_validation_config

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name,
        )

        return data_transformation_config

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            eval_strategy=params.eval_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps,
        )

        return model_trainer_config

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])
        model_eval_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name,
        )

        return model_eval_config


### Components

###### Data Ingestion Component

In [ ]:
import os
from pathlib import Path
import urllib.request as request
import zipfile

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL, filename=self.config.local_data_file
            )
        else:
            print(
                f"File already exists of size: {get_size(Path(self.config.local_data_file))}"
            )

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)


###### Data Preparation Component

In [ ]:
from datasets import load_dataset, concatenate_datasets
from pathlib import Path


class DataPreparation:
    def __init__(self, config: DataPreparationConfig):
        self.config = config

    def main(self):
        config = self.config

        train_ds_path = Path(f"{config.data_path}/{config.train_filename}")
        validation_ds_path = Path(f"{config.data_path}/{config.validation_filename}")
        test1_ds_path = Path(f"{config.data_path}/{config.test_filenames[0]}")
        test2_ds_path = Path(f"{config.data_path}/{config.test_filenames[1]}")

        dataset = load_dataset(
            "csv",
            data_files={
                "train": str(train_ds_path),
                "validation": str(validation_ds_path),
                "test_1": str(test1_ds_path),
                "test_2": str(test2_ds_path),
            },
        )

        dataset["test"] = concatenate_datasets([dataset["test_1"], dataset["test_2"]])

        dataset.pop("test_1")
        dataset.pop("test_2")

        dataset.save_to_disk(config.root_dir)

###### Data Validation Component

In [ ]:
import os

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_exists(self) -> bool:
        try:
            validation_status = None

            all_files = os.listdir(
                os.path.join("drive", "MyDrive", "Text-summerizer", "artifacts", "data_preparation")
            )

            for file in all_files:
                if file not in self.config.ALL_REQUIRED_FILES:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")


            return validation_status

        except Exception as e:
            raise e

###### Data Transformation Component

In [ ]:
import os
from transformers import AutoTokenizer
from datasets import load_from_disk

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

        # Configure for Pegasus
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def convert_examples_to_features(self, example_batch):
        input_encondings = self.tokenizer(
            example_batch["dialogue"], max_length=1024, truncation=True
        )


        target_encondings = self.tokenizer(
            example_batch["summary"], max_length=128, truncation=True
        )

        return {
            "input_ids": input_encondings["input_ids"],
            "attention_mask": input_encondings["attention_mask"],
            "labels": target_encondings["input_ids"],
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features, batched=True
        )
        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )


###### Model Trainer Component

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
import torch


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            print("✅ Set pad_token to eos_token for Pegasus")

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        model_pegasus.config.tie_word_embeddings = False

        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus, padding=True)

        # loading data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=1e6,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            save_total_limit=3,  # Keep only last 3 checkpoints
            load_best_model_at_end=True,  # Load best model at end
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            # Fix for pin_memory warning
            dataloader_pin_memory=torch.cuda.is_available(),
            # Save best model
            save_strategy="steps",  # Match eval_strategy
        )

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            # tokenizer=tokenizer,
            processing_class=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"],
        )

        trainer.train()

        ## Save model
        model_pegasus.save_pretrained(
            os.path.join(self.config.root_dir, "pegasus-samsum-model")
        )
        ## Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


###### Model Evaluation Component

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
from evaluate import load as load_metric
import torch
import pandas as pd
from tqdm import tqdm

class ModelEvaluation:
  def __init__(self, config: ModelEvaluationConfig):
    self.config = config

  def generate_batch_sized_chunks(self, list_of_elements, batch_size):
    for i in range(0, len(list_of_elements), batch_size):
      yield list_of_elements[i : i + batch_size]

  def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer,
                                  batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu",
                                  column_text="article",
                                  column_summary="highlights"):
    article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024,  truncation=True, padding="max_length", return_tensors="pt")

        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                  attention_mask=inputs["attention_mask"].to(device),
                                  length_penalty=0.8, num_beams=8, max_length=128)

        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True) for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]


        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #  Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score

  def evaluate(self):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
    model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

    dataset_samsum_pt = load_from_disk(self.config.data_path)

    rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
    rouge_metric = load_metric("rouge")

    score = self.calculate_metric_on_test_ds(
                  dataset_samsum_pt["test"][0:10],
                  rouge_metric, model_pegasus, tokenizer,
                  batch_size=2, column_text='dialogue', column_summary='summary'
                )

    rouge_dict = dict((rn, score[rn]) for rn in rouge_names)

    data_frame = pd.DataFrame(rouge_dict, index=['pegasus'])
    data_frame.to_csv(self.config.metric_file_name, index=False)

### Piplines

Responsable for running model from scratch

```
1. Data Ingestion Training Pipline
2. Data Validation Training Pipline
3. Data Transformation Training Pipline
4. Model Trainer Training Pipline
5. Model Evaluation Training Pipline
```

In [ ]:
class ProjectPiplines:
  def __init__(self):
    self.config = ConfigrationManager()

  def run_data_ingestion_pipline(self):
    data_ingestion_config = self.config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

  def run_data_validation_pipline(self):
    data_validation_config = self.config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_files_exists()

  def run_data_transformation_pipline(self):
    data_tranfromation_config = self.config.get_data_transformation_config()
    data_tranfromation = DataTransformation(config=data_tranfromation_config)
    data_tranfromation.convert()

  def run_model_trainer_pipline(self):
    model_trainer_config = self.config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()

  def run_model_evaluation_pipline(self):
    model_eval_config = self.config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_eval_config)
    model_evaluation.evaluate()

### Main Project

###### Main Class

In [ ]:
class Main:
  def __init__(self):
    self.pipline = ProjectPiplines()

  def run_all(self):
    self.run_data_ingestion_process()
    self.run_data_validation_process()
    self.run_data_transformation_process()
    self.run_model_trainer_process()

  def run_data_ingestion_process(self):
    try:
      print(">>>>>> Process {Data Ingestion} started <<<<<<<")
      data_ingestion = self.pipline.run_data_ingestion_pipline()
      print(">>>>>> Process {Data Ingestion} ended <<<<<<<")
    except Exception as e:
      print(e) ## Only for development
      raise e

  def run_data_validation_process(self):
    try:
      print(">>>>>> Process {Data Validation} started <<<<<<<")
      data_validation = self.pipline.run_data_validation_pipline()
      print(">>>>>> Process {Data Validation} ended <<<<<<<")
    except Exception as e:
      print(f"Unexpected exception: {e}")
      raise e

  def run_data_transformation_process(self):
    try:
      print(">>>>>> Process {Data Tranformation} started <<<<<<<")
      data_tranformation = self.pipline.run_data_transformation_pipline()
      print(">>>>>> Process {Data Validation} ended <<<<<<<")
    except Exception as e:
      print(f"Unexpected exception: {e}") ## Only for development
      raise e

  def run_model_trainer_process(self):
    try:
      print(">>>>>> Process {Model Trainer} started <<<<<<<")
      model_trainer = self.pipline.run_model_trainer_pipline()
      print(">>>>>> Process {Model Trainer} ended <<<<<<<")
    except Exception as e:
      print(f"Unexpected exception: {e}") ## Only for development
      raise e

  def run_model_evaluation_process(self):
    try:
      print(">>>>>> Process {Model Evaluation} started <<<<<<<")
      model_eval = self.pipline.run_model_evaluation_pipline()
      print(">>>>>> Process {Model Evaluation} ended <<<<<<<")
    except Exception as e:
      print(f"Unexpected exception: {e}") ## Only for development
      raise e

In [ ]:
main = Main()

###### Data Ingestion Stage

In [ ]:
main.run_data_ingestion_process()

>>>>>> Process {Data Ingestion} started <<<<<<<
File already exists of size: ~ 7718 KB
>>>>>> Process {Data Ingestion} ended <<<<<<<


###### Data Validation Stage

In [ ]:
main.run_data_validation_process()

>>>>>> Process {Data Validation} started <<<<<<<
>>>>>> Process {Data Validation} ended <<<<<<<


###### Data Transformation Stage

In [ ]:
main.run_data_transformation_process()

>>>>>> Process {Data Tranformation} started <<<<<<<


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

>>>>>> Process {Data Validation} ended <<<<<<<


###### Model Trainer Stage

In [ ]:
# Add at the very top of your training script
import os

# Memory optimization environment variables
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OMP_NUM_THREADS"] = "4"  # Limit CPU threads
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Reduce memory fragmentation

# Then run your training

In [ ]:
main.run_model_trainer_process()

>>>>>> Process {Model Trainer} started <<<<<<<


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Unexpected exception: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.27 GiB is allocated by PyTorch, and 147.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.27 GiB is allocated by PyTorch, and 147.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

###### Model Evaluation Stage

In [ ]:
main.run_model_evaluation_process()

### Application Runner

In [ ]:
# class Application:
#   pass